# HIPE-2026 Strategy 2: End-to-End Fine-Tuning

This notebook provides the environment and execution steps to fine-tune the XLM-RoBERTa model for the HIPE-2026 Relation Extraction task using a free T4 GPU on Google Colab.

### 1. Setup Environment

In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn tqdm
import torch
print(f"Using GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

### 2. Clone Repository and Data

> [!NOTE]
> If your repository is **private**, you will need a **GitHub Personal Access Token (PAT)**. You can generate one in [Settings > Developer Settings > Personal Access Tokens](https://github.com/settings/tokens).

In [ ]:
import getpass
import os
import shutil

repo_name = "HIPE-2026-Team-Hansel-Gretel"
repo_url = f"https://github.com/ikshv4ku/{repo_name}.git"

%cd /content

if not os.path.exists(repo_name):
    token = getpass.getpass('Paste your GitHub Personal Access Token (PAT):')
    !git clone https://{token}@github.com/ikshv4ku/{repo_name}.git
else:
    print(f"{repo_name} already exists. Pulling latest changes...")
    %cd {repo_name}
    !git pull origin main
    %cd /content

%cd /content/{repo_name}

# Force cleanup of broken submodule directory to allow re-clone
if os.path.exists('HIPE-2026-data') and not os.path.exists('HIPE-2026-data/data/newspapers/v1.0'):
    print("Cleaning up broken submodule directory...")
    !rm -rf HIPE-2026-data

!git submodule update --init --recursive

# Verification: check if data exists
data_check_path = "HIPE-2026-data/data/newspapers/v1.0"
if os.path.exists(data_check_path) and len(os.listdir(data_check_path)) > 0:
    print("✅ Data submodule correctly initialized!")
else:
    print("❌ Data missing. Manually cloning submodule...")
    !rm -rf HIPE-2026-data
    !git clone https://github.com/hipe-eval/HIPE-2026-data.git HIPE-2026-data

### 3. Generate Multilingual Data Splits

Since the data splits (`train-all.jsonl`, `dev-all.jsonl`) are generated files, we need to create them in the Colab environment before training starts.

In [ ]:
!python3 strategy1.5/scripts/prepare_multilingual_split.py

### 4. Launch Strategy 2 Training

This will run the fine-tuning for 10 epochs. The best model will be saved to `strategy2/results/models/best_ft_model.pt`.

In [ ]:
import sys
sys.path.append('./strategy2/scripts')
!export PYTHONPATH=$PYTHONPATH:$(pwd)/strategy2/scripts && python3 strategy2/scripts/train.py --epochs 10 --batch_size 8

### 5. Run Inference and Analysis

In [ ]:
!python3 strategy2/scripts/inference.py --input HIPE-2026-data/data/newspapers/v1.0/splits/HIPE-2026-v1.0-impresso-dev-all.jsonl --output strategy2/results/predictions/preds-dev-all.jsonl
!python3 strategy2/scripts/analyse_results.py